In [ ]:
import dspy
import os
import json
from tqdm import tqdm
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module='transformers')

try:
    lm = dspy.LM('xai/grok-3-mini')
    dspy.configure(lm=lm)
    print("LLM for evaluation configured successfully.")
except Exception as e:
    print(f"Could not configure LLM, evaluation might fail. Error: {e}")
    print("Please ensure you have a valid API key set for your chosen model.")

LLM for evaluation configured successfully.


In [ ]:
def load_pragmaticqa_first_questions(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            conversation = json.loads(line)
            if not conversation['qas']:
                continue
            
            first_qa = conversation['qas'][0]
            
            literal_context = " ".join([span['text'] for span in first_qa['a_meta'].get('literal_obj', [])])
            pragmatic_context = " ".join([span['text'] for span in first_qa['a_meta'].get('pragmatic_obj', [])])
            
            data.append({
                'question': first_qa['q'],
                'gold_answer': first_qa['a'],
                'literal_context': literal_context.strip(),
                'pragmatic_context': pragmatic_context.strip()
            })
    return data

val_data_path = '../PragmatiCQA/data/val.jsonl'
val_data = load_pragmaticqa_first_questions(val_data_path)

print(f"Loaded {len(val_data)} first-question examples from {val_data_path}")
print("\nExample data point:")
print(json.dumps(val_data[0], indent=2))

Loaded 179 first-question examples from ../PragmatiCQA/data/val.jsonl

Example data point:
{
  "question": "who is freddy krueger?",
  "gold_answer": "Freddy Kruger is the nightmare in nighmare on Elm street. Please note, and to be very clear, the system that loads up wiki is not allowing access to Adam Prag, to the page... so I'll have to go from memory.  Normally you can paste things and back up what you are saying, but today that's not happening. alas.",
  "literal_context": "Cannot GET /wiki/A%20N",
  "pragmatic_context": "Cannot GET /wiki/A%20N"
}


In [ ]:
import os
from glob import glob
from bs4 import BeautifulSoup
from tqdm import tqdm

def read_corpus_from_sources(sources_path):
   
    search_pattern = os.path.join(sources_path, '*', '*.html')
    
    html_files = glob(search_pattern)
    
    if not html_files:
        raise FileNotFoundError(f"No HTML files were found using the pattern: {search_pattern}. Please verify the 'sources_path' is correct.")
        
    print(f"Found {len(html_files)} HTML files to process.")
    
    # Process each file and extract its text
    corpus_texts = []
    for file_path in tqdm(html_files, desc="Reading files"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                soup = BeautifulSoup(f, 'html.parser')
                corpus_texts.append(soup.get_text())
        except Exception as e:
            print(f"Warning: Could not read file {file_path}. Error: {e}")
            
    return corpus_texts

In [ ]:
try:
    corpus_path = '/Users/poslevkusie/BGU/NLP/hw3/PragmatiCQA-sources/pragmaticqa_corpus'
    full_corpus = read_corpus_from_sources(corpus_path)
    print(f"Successfully loaded {len(full_corpus)} total documents.")

    num_docs_to_use = len(full_corpus) // 2
    
    # Slice the list to get only the first half, otherwise computer cant process the data set
    corpus = full_corpus[:num_docs_to_use]
    print(f"Using the first half of the dataset: {len(corpus)} documents.")
    # -----------------------------------------------

    model = SentenceTransformer("sentence-transformers/static-retrieval-mrl-en-v1", device="cpu")
    embedder = dspy.Embedder(model.encode)

    topk_docs_to_retrieve = 5
    search = dspy.retrievers.Embeddings(embedder=embedder, corpus=corpus, k=topk_docs_to_retrieve)
    
    print(f"\n Retriever 'search' object created successfully.")

except Exception as e:
    print(f"\n An error occurred during retriever setup: {e}")

Found 28570 HTML files to process.


Reading files: 100%|██████████| 28570/28570 [02:14<00:00, 213.02it/s]


Successfully loaded 28570 total documents.
Using the first half of the dataset: 14285 documents.

✅ Retriever 'search' object created successfully.


In [ ]:
class HuggingFaceQA(dspy.Module):
    def __init__(self, model_name):
        super().__init__()
        self.qa_pipeline = pipeline(
            "question-answering", 
            model=model_name, 
            tokenizer=model_name
        )
        self.predict = self 
    def forward(self, question, context):
        if not context or not question:
            return dspy.Prediction(response="")
        
        max_len = 512 * 5 # A heuristic length limit
        truncated_context = context[:max_len]
        
        result = self.qa_pipeline(question=question, context=truncated_context)
     
        return dspy.Prediction(response=result['answer'])

qa_model = HuggingFaceQA(model_name='distilbert-base-cased-distilled-squad')


Device set to use mps:0


In [ ]:
from dspy.evaluate import Evaluate
from dspy.evaluate import SemanticF1

def prepare_dataset(data, context_key):
    dataset = []
    for item in data:
        if item[context_key]:
            example = dspy.Example(
                question=item['question'], 
                context=item[context_key],
                response=item['gold_answer'] 
            ).with_inputs('question', 'context')
            dataset.append(example)
    return dataset

semantic_f1_metric = SemanticF1()

evaluator = Evaluate(devset=[], metric=semantic_f1_metric, num_threads=1, display_progress=True, display_table=5)

In [7]:
print("--- Evaluating on Literal Context ---")
literal_devset = prepare_dataset(val_data, 'literal_context')
evaluator.devset = literal_devset
literal_scores = evaluator(qa_model)
print("\nResults for Literal Context:")
print(literal_scores)

--- Evaluating on Literal Context ---
Average Metric: 74.21 / 179 (41.5%): 100%|██████████| 179/179 [00:05<00:00, 32.90it/s]

2025/08/17 15:01:53 INFO dspy.evaluate.evaluate: Average Metric: 74.21161437274885 / 179 (41.5%)


,question,context,example_response,pred_response,SemanticF1
0,who is freddy krueger?,Cannot GET /wiki/A%20N,Freddy Kruger is the nightmare in nighmare on Elm street. Please n...,Cannot GET /wiki/A%20N,
1,who was the star on this movie?,Cannot GET /wiki/A%20Nightmare%20on%20Elm%20Street/A%20Nightmare%2...,"Robert Englund IS Freddy Kruger, the bad guy for these films. Note...",20Nightmare,
2,What is the movie about?,Cannot GET /wiki/A%20Nightmare%20on%20Elm%20Street/A%20Nightmare%2...,"Ok, here goes, I'm getting ""Cannot get""..so, Nightmare on Elm stre...",20film,
3,Who directed the new film?,Cannot GET /wiki/A%20Nightmare%20on%20Elm%20Street/A%20Nightmare%2...,It was Directed by: Samuel Bayer. Note that the link here is broke...,2010%20film,
4,Is the Batman comic similar to the movies?,"Bruce Wayne is born to Dr. Thomas Wayne and his wife Martha Kane ,...","I would say the movie and comics has same story line, as Batmans p...",Gotham City socialites,✔️ [0.400]



Results for Literal Context:
EvaluationResult(score=41.46, results=<list of 179 results>)


In [8]:
print("\n--- Evaluating on Pragmatic Context ---")
pragmatic_devset = prepare_dataset(val_data, 'pragmatic_context')
evaluator.devset = pragmatic_devset
pragmatic_scores = evaluator(qa_model)
print("\nResults for Pragmatic Context:")
print(pragmatic_scores)


--- Evaluating on Pragmatic Context ---
Average Metric: 66.87 / 179 (37.4%): 100%|██████████| 179/179 [00:03<00:00, 47.79it/s]

2025/08/17 15:01:57 INFO dspy.evaluate.evaluate: Average Metric: 66.87342965150233 / 179 (37.4%)


,question,context,example_response,pred_response,SemanticF1
0,who is freddy krueger?,Cannot GET /wiki/A%20N,Freddy Kruger is the nightmare in nighmare on Elm street. Please n...,Cannot GET /wiki/A%20N,
1,who was the star on this movie?,Cannot GET /wiki/A%20Nightmare%20on%20Elm%20Street/A%20Nightmare%2...,"Robert Englund IS Freddy Kruger, the bad guy for these films. Note...",20Nightmare,
2,What is the movie about?,Cannot GET /wiki/A%20Nightmare%20on%20Elm%20Street/A%20Nightmare%2...,"Ok, here goes, I'm getting ""Cannot get""..so, Nightmare on Elm stre...",20film,
3,Who directed the new film?,Cannot GET /wiki/A%20Nightmare%20on%20Elm%20Street/A%20Nightmare%2...,It was Directed by: Samuel Bayer. Note that the link here is broke...,2010%20film,
4,Is the Batman comic similar to the movies?,"While returning home one night, his parents were killed by a small...","I would say the movie and comics has same story line, as Batmans p...",his parents were killed by a small-time criminal named Joe Chill,✔️ [0.400]



Results for Pragmatic Context:
EvaluationResult(score=37.36, results=<list of 179 results>)


In [ ]:
print("\n--- Evaluating on Retrieved Context ---")
retrieved_devset = []
for item in tqdm(val_data, desc="Retrieving context for questions"):
    retrieved_passages = search(item['question']).passages
    retrieved_context = " \n\n ".join(retrieved_passages)
    
    if retrieved_context:
        example = dspy.Example(
            question=item['question'],
            context=retrieved_context,
            response=item['gold_answer'] 
        ).with_inputs('question', 'context')
        retrieved_devset.append(example)

evaluator.devset = retrieved_devset
retrieved_scores = evaluator(qa_model)
print("\nResults for Retrieved Context:")
print(retrieved_scores)


--- Evaluating on Retrieved Context ---


Retrieving context for questions: 100%|██████████| 179/179 [00:22<00:00,  7.80it/s]


Average Metric: 17.08 / 179 (9.5%): 100%|██████████| 179/179 [5:40:40<00:00, 114.19s/it]     

2025/08/17 20:46:48 INFO dspy.evaluate.evaluate: Average Metric: 17.076466954379104 / 179 (9.5%)


,question,context,example_response,pred_response,SemanticF1
0,who is freddy krueger?,Freddy Krueger General information Age ? (at the time of physical ...,Freddy Kruger is the nightmare in nighmare on Elm street. Please n...,a pedophile and child molester,
1,who was the star on this movie?,Starling Family Mother Gary Meyers Biographical information Race T...,"Robert Englund IS Freddy Kruger, the bad guy for these films. Note...",Richard Kind\n \n\n\n\n\n\n\n\n Dr. Gary Meyers,
2,What is the movie about?,Skulduggery Pleasant (movie) left This article is about unreleased...,"Ok, here goes, I'm getting ""Cannot get""..so, Nightmare on Elm stre...",unreleased content,
3,Who directed the new film?,Stuart Gordon Full Name Stuart Alan Gordon Connection to the Mytho...,It was Directed by: Samuel Bayer. Note that the link here is broke...,Erica Milsom,
4,Is the Batman comic similar to the movies?,Warner Bros. is the film studio that owns DC Comics and the Batman...,"I would say the movie and comics has same story line, as Batmans p...",The subsidiary returned after a two-year hiatus in 2010.,



Results for Retrieved Context:
EvaluationResult(score=9.54, results=<list of 179 results>)
